In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# LongSplat 환경 세팅 (Colab A100)

> **[NVlabs/LongSplat](https://github.com/NVlabs/LongSplat)** — ICCV 2025  
> Colab A100 환경 기준 (CUDA 12.1 / PyTorch 2.5.1)

---

## ✅ 순서
1. GPU 확인
2. Git Clone
3. CUDA / GCC 세팅
4. PyTorch 및 패키지 설치
5. Submodule 빌드
6. 데이터셋 업로드
7. 학습 실행


## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. Git Clone

In [ ]:
!git clone --recursive https://github.com/NVlabs/LongSplat.git
%cd /content/LongSplat

## 2. CUDA 12.1 PATH 설정 및 GCC 9 설치

In [ ]:
import os
os.environ["PATH"] = "/usr/local/cuda-12.1/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda-12.1/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

In [ ]:
!gcc --version

In [ ]:
!apt-get install -y gcc-9 g++-9 -q
!update-alternatives --install /usr/bin/gcc gcc /usr/bin/gcc-9 9
!update-alternatives --install /usr/bin/g++ g++ /usr/bin/g++-9 9
!update-alternatives --set gcc /usr/bin/gcc-9
!update-alternatives --set g++ /usr/bin/g++-9
!gcc --version

## 3. PyTorch 2.5.1 + CUDA 12.1 설치

In [ ]:
!pip install torch==2.5.1 torchvision --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

## 4. pytorch3d & torch_scatter 설치 (pre-built wheel)

In [ ]:
!pip install --extra-index-url https://miropsota.github.io/torch_packages_builder \
    pytorch3d==0.7.8+pt2.5.1cu121 -q

In [ ]:
!pip install torch_scatter -f https://data.pyg.org/whl/torch-2.5.1+cu121.html -q

## 5. requirements.txt 설치

In [ ]:
!grep -v "pytorch3d" /content/LongSplat/requirements.txt > /tmp/req_no_pytorch3d.txt
!pip install -r /tmp/req_no_pytorch3d.txt

## 6. Submodule 빌드

In [ ]:
import re

path = "/content/LongSplat/submodules/fused-ssim/setup.py"
with open(path, "r") as f:
    content = f.read()

print(content)  # 내용 확인, -arch=sm_89 부분을 GPU에 맞게 수정

content_fixed = content.replace("sm_89", "sm_80").replace("8.9", "8.0")

with open(path, "w") as f:
    f.write(content_fixed)

print("수정 완료")
print(open(path).read())

In [ ]:
# A100: Compute Capability 8.0
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "8.0"

%cd submodules/fused-ssim
!pip install -e . --no-build-isolation
%cd /content/LongSplat

In [ ]:
!pip install submodules/simple-knn --no-build-isolation

In [ ]:
!pip install submodules/diff-gaussian-rasterization --no-build-isolation

In [ ]:
!rm -rf /content/LongSplat/submodules/fused-ssim/build
!rm -f /content/LongSplat/submodules/fused-ssim/fused_ssim/*.so
!cd /content/LongSplat/submodules/fused-ssim && python setup.py build_ext --inplace 2>&1
!pip install /content/LongSplat/submodules/fused-ssim --force-reinstall 2>&1

In [ ]:
%cd /content/LongSplat

## 7. (선택) RoPE CUDA 커널 컴파일

런타임 속도 향상을 위한 선택 사항입니다.

In [ ]:
!cd submodules/mast3r/dust3r/croco/models/curope/
!python setup.py build_ext --inplace
!cd /content/LongSplat

## 8. 학습 실행

파라미터를 수정한 뒤 셀을 다시 실행하면 스크립트가 덮어씌워지고 바로 학습이 시작됩니다.

In [ ]:
import subprocess
result = subprocess.run(
    ["find", "/content/drive/MyDrive/Research/Dataset", "-name", ".DS_Store", "-delete"],
    capture_output=False, text=True
)
print("삭제 완료")

In [ ]:
import os, random, textwrap, subprocess, sys
from datetime import datetime, timezone, timedelta
from pathlib import Path

from tqdm.notebook import tqdm
import tqdm as tqdm_module
tqdm_module.std.tqdm = tqdm

KST = timezone(timedelta(hours=9))

SCENES = ["grass_snow", "grass_rain",
          "hydrant_snow", "hydrant_rain",
          "pillar_snow", "pillar_rain",
          "road_snow", "road_rain",
          "sky_snow", "sky_rain",
          "stair_snow", "stair_rain"]

for scene in SCENES:
    input_path  = '/content/drive/MyDrive/Research/Dataset/free_outdoor_weathered/weathered'
    r           = 1
    pose_iter   = 100
    local_iter  = 200
    global_iter = 500
    port        = random.randint(10000, 30000)
    timestamp   = datetime.datetime.now(KST).strftime("%Y-%m-%d_%H:%M")
    outputs     = f"/content/drive/MyDrive/Research/outputs/{scene}/{timestamp}"

    script = textwrap.dedent(f"""
        #!/bin/bash
        ulimit -n 4096
        cd /content/LongSplat
        python train.py --eval -s {input_path} -m {outputs} --port {port} \\
            --images {scene} --mode custom -r {r} \\
            --pose_iteration {pose_iter} --local_iter {local_iter} --global_iter {global_iter}
        python render.py  -m {outputs}
        python metrics.py -m {outputs}
    """).strip()

    script_path = Path("/content/LongSplat/scripts/train_custom.sh")
    script_path.write_text(script, encoding="utf-8")
    print(f"\n{'='*60}\nScene: {scene}\n{script}\n{'='*60}")

    with subprocess.Popen(
        ["bash", str(script_path)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"}
    ) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)

    if proc.returncode != 0:
        raise RuntimeError(f"[{scene}] Script failed (exit code {proc.returncode})")

    print(f"\n[{scene}] ✓ 완료")